# 1. Basic concepts

A composite is **a number that carries its own metadata**. The value sits at grade 0;
everything else records how the number came to be.

The whole system follows from one refusal: **zero does not annihilate**. Multiply by
zero and the operand is not lost, it moves down a grade. Subtract a number from
itself and what cancelled is recorded. Divide by zero and the calculation continues.

Euler argued for this in 1755 (*Institutiones calculi differentialis*, §85): different
zeros need different characters, or the ratio between them is lost.

In [1]:
import warnings
import composite.composite_lib as cl
from composite import R, ZERO, INF, Composite, display
from composite.backends.config import use_dict

use_dict(); cl._refresh_constants()          # the dict backend holds every grade kind
# MAX_ACTIVE_DIMS is left at its default of 60. Raising it removes the
# truncation guard, and an iterative algorithm then grows terms without
# bound -- solve_ode below did not finish 20 steps in 30s at 10**9, and
# finishes 200 steps in 0.86s at the default.

R(3) + ZERO                                   # importing composite installs a grade table

grade,coefficient,meaning
0,3,the value
-1,1,d(1) = 1


## The grade axis is a rate

`|c|_d` means coefficient `c` at grade `d`, which is `c * h**(-d)`.

* **negative grades vanish** - a term at grade `-k` goes to zero like `h**k`
* **grade 0 is the ordinary value**
* **positive grades blow up** - grade `+k` diverges like `h**-k`

Inversion flips the axis through 0, which is why division is total.

In [2]:
for label, value in (("h, the infinitesimal", ZERO),
                     ("h squared", ZERO * ZERO),
                     ("an ordinary 5", R(5)),
                     ("1/h, an infinity", INF),
                     ("1/h squared", INF * INF)):
    print("%-22s %-12s lead_order %s" % (label, value, value.lead_order()))

h, the infinitesimal   <|1|₋₁>      lead_order 1.0
h squared              <|1|₋₂>      lead_order 2.0
an ordinary 5          <|5|₀>       lead_order 0.0
1/h, an infinity       <|1|₁>       lead_order -1.0
1/h squared            <|1|₂>       lead_order -2.0


## The zero rules

These are what the library actually prints, not a paraphrase.

In [3]:
warnings.simplefilter("ignore")               # the library warns on each of these; see below
for w in (cl.CancellationWarning, cl.NotConventionalWarning):
    warnings.filterwarnings("ignore", category=w)

rules = [("1 - 1",        lambda: R(1) - R(1)),
         ("6 - 6",        lambda: R(6) - R(6)),
         ("(-6) - (-6)",  lambda: R(-6) - R(-6)),
         ("5 + 0",        lambda: R(5) + R(0)),
         ("0 - 0",        lambda: R(0) - R(0)),
         ("0 / 0",        lambda: R(0) / R(0)),
         ("1 / 0",        lambda: R(1) / R(0)),
         ("7*0*0/0/0",    lambda: R(7) * R(0) * R(0) / R(0) / R(0))]
for label, fn in rules:
    print("%-14s ->  %s" % (label, fn()))

1 - 1          ->  <|1|₋₁>
6 - 6          ->  <|6|₋₁>
(-6) - (-6)    ->  <|-6|₋₁>
5 + 0          ->  <|5|₀ |1|₋₁>
0 - 0          ->  <|1|₋₂>
0 / 0          ->  <|1|₀>
1 / 0          ->  <|1|₁>
7*0*0/0/0      ->  <|7|₀>


The last line is the design goal: multiply by zero twice, divide by zero twice,
and 7 comes back. Nothing was destroyed on the way.

`a - a = a*h` - the residue is the **whole annihilated quantity**, one grade down.
That is what makes the next cell work.

## Euler's point: zeros have ratios

If every zero were the same zero, every ratio between two of them would be 1.
Euler's §85 says that must not happen, and here it doesn't.

In [4]:
x, y = R(3), R(2)
print("(2-2)/(3-3)   =", (R(2) - R(2)) / (R(3) - R(3)), "   want 2/3")
print("(6-6)/(2-2)   =", (R(6) - R(6)) / (R(2) - R(2)), "   want 3")
print("(x-x)/(y-y)   =", (x - x) / (y - y), "   want x/y = 1.5")

(2-2)/(3-3)   = <|0.666667|₀>    want 2/3
(6-6)/(2-2)   = <|3|₀>    want 3
(x-x)/(y-y)   = <|1.5|₀>    want x/y = 1.5


## Derivatives fall out of the arithmetic

Evaluate at `a + h` and the coefficient at grade `-k` is the k-th Taylor coefficient.
No separate machinery, no step size, no difference quotient.

In [5]:
seed = R(3) + ZERO          # x = 3 + h
value = seed * seed         # x squared
value

grade,coefficient,meaning
0,9,the value
-1,6,d(1) = 6
-2,1,d(2) = 2


In [6]:
print("f(3)       st()  =", value.st())
print("f'(3)      d(1)  =", value.d(1))
print("f''(3)     d(2)  =", value.d(2))
print("f'''(3)    d(3)  =", value.d(3))
print()
print("one evaluation, every order, exact.")

f(3)       st()  = 9.0
f'(3)      d(1)  = 6.0
f''(3)     d(2)  = 2.0
f'''(3)    d(3)  = 0.0

one evaluation, every order, exact.


## NOTHING is not zero

`Composite({})` is **absence** - there is no term to read. A zero is a *value*, and
in this system a value that happens to be zero is an infinitesimal. The two are
different objects and the library keeps them apart.

In [7]:
nothing, written_zero, infinitesimal = Composite({}), R(0), ZERO
for label, v in (("Composite({})  NOTHING",    nothing),
                 ("R(0)  a written zero",      written_zero),
                 ("ZERO  the infinitesimal",   infinitesimal)):
    print("%-26s %-10s terms %s" % (label, v, v.coeffs_dict()))

print()
print("NOTHING has no term at all. R(0) has a term whose value is 0 -- an")
print("EXPRESSED zero, which R1 converts the moment it is used as an operand.")
print("The difference shows in what they do, not in how they print:")
print("  R(5) + NOTHING =", R(5) + nothing, "  <- absence adds no order")
print("  R(5) + R(0)    =", R(5) + R(0), "  <- the written zero converted")
print()
print("and both are still equal to the infinitesimal, because R1 reads them so:")
print("  R(0)  == ZERO ?", R(0) == ZERO, "   NOTHING == ZERO ?", nothing == ZERO)

Composite({})  NOTHING     ∅          terms {}
R(0)  a written zero       <|0|₀>     terms {0: 0.0}
ZERO  the infinitesimal    <|1|₋₁>    terms {-1: 1.0}

NOTHING has no term at all. R(0) has a term whose value is 0 -- an
EXPRESSED zero, which R1 converts the moment it is used as an operand.
The difference shows in what they do, not in how they print:
  R(5) + NOTHING = <|5|₀>   <- absence adds no order
  R(5) + R(0)    = <|5|₀ |1|₋₁>   <- the written zero converted

and both are still equal to the infinitesimal, because R1 reads them so:
  R(0)  == ZERO ? True    NOTHING == ZERO ? False


## What it costs

The price is exact and it is a theorem, not a preference. In anything with an
additive identity, cancellation and distributivity, annihilation is **forced**:

    0*x = (0 + 0)*x = 0*x + 0*x,  cancel, and 0*x = 0

So `0*x = 0` is not a fact about zero - it is a consequence of three assumptions.
This system gives up the additive identity. `5 + 0` is not 5.

A second, smaller cost: addition stops commuting on a cancelling pair.

In [8]:
print("R(2) + R(-2)  =", R(2) + R(-2))
print("R(-2) + R(2)  =", R(-2) + R(2), "  <- differs by a sign")
print()
print("but multiplication is untouched, and distributivity across a cancellation holds:")
a, b, c = R(3), R(2), R(2)
print("  a*(b-c)      =", a * (b - c))
print("  a*b - a*c    =", a * b - a * c)

R(2) + R(-2)  = <|2|₋₁>
R(-2) + R(2)  = <|-2|₋₁>   <- differs by a sign

but multiplication is untouched, and distributivity across a cancellation holds:
  a*(b-c)      = <|6|₋₁>
  a*b - a*c    = <|6|₋₁>


## Summary

| | |
|---|---|
| `a - a` | `a*h` - the quantity, one grade down |
| `c + 0` | not `c` - a zero was expressed |
| `1 / 0` | `\|1\|₁` - an infinity of definite order |
| `0 / 0` | `\|1\|₀` - two zeros of the same order have ratio 1 |
| grade `-k` | vanishes like `h**k`; also the k-th Taylor coefficient |
| grade `+k` | diverges like `h**-k` |
| `Composite({})` | absence, outside the numbers |

Next: **2. Simple usage** - what to actually do with this.